# Meeting 21 · Data and validation

**Task:** at the start of each day, predict that day’s total bike rentals. Assume yesterday’s rental count is finalized and available.

We will define the available inputs, reserve later data, fit a baseline, and compare predictions using an explicit error measure. The notebook uses the daily Capital Bikeshare records from 2011–2012. It is a historical teaching experiment; it does not model the number of bicycles required at individual stations.

**In class:** complete Sections 1–4 and the decision record. The later exercises extend the same ideas. No final-test scores are needed today.

**Environment:** Python 3 with NumPy, pandas, matplotlib, and scikit-learn (available in Colab). The data are embedded below, so no data download is needed. Run the notebook from top to bottom.

**Data credit:** Fanaee-T, H. (2013), [Bike Sharing](https://doi.org/10.24432/C5W894), UCI Machine Learning Repository, [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The original daily file is preserved. We add a one-day lag and remove the first row, which has no preceding day in the file.

In [ ]:
import io, base64, zlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 12})

In [ ]:
# Original UCI day.csv, compressed only to keep this notebook self-contained.
DATA_B64 = "..."
raw = pd.read_csv(io.BytesIO(zlib.decompress(base64.b64decode(DATA_B64))),
                  parse_dates=["dteday"]).sort_values("dteday").reset_index(drop=True)
assert len(raw) == 731
assert raw.dteday.is_unique
assert raw.dteday.diff().dropna().eq(pd.Timedelta(days=1)).all()
print("Loaded 731 consecutive daily records.")

## 1. Define the examples and their available inputs

`cnt` is today’s total rentals. `workingday` is 1 for a day that is neither a weekend nor a holiday. `yesterday_rentals` is the preceding day’s total, constructed after sorting by date.

Our first two features are available at the prediction time. Today’s final casual and registered rental counts, and today’s observed daily weather, are not. A weather forecast issued beforehand would be a different input and would require its own data source.

In [ ]:
frame = raw.assign(yesterday_rentals=raw["cnt"].shift(1)).dropna().copy()
features = ["yesterday_rentals", "workingday"]
target = "cnt"
frame.loc[frame.dteday < "2012-01-01", ["dteday", *features, target]].head()

**Your reasoning 1.** Choose one allowed input and one disallowed input. Explain the timing of each. Why is correlation with the target insufficient to decide whether a feature is usable?

*Write your answer here.*

## 2. Split before fitting

The question concerns future days, so we respect time. Training uses January 2–December 31, 2011. Validation uses January–June 2012. July–December 2012 stays reserved.

These are successive **one-day-ahead** predictions made with a fixed fitted model. When a validation day ends, its observed total can become the next day’s lagged input. The model’s fitted coefficients do not change. We are not forecasting six months of unknown daily counts all at once.

In [ ]:
train = frame.loc[frame.dteday < "2012-01-01"].copy()
valid = frame.loc[(frame.dteday >= "2012-01-01") & (frame.dteday < "2012-07-01")].copy()
# Keep only test inputs here; do not inspect or score test targets during development.
X_test_reserved = frame.loc[frame.dteday >= "2012-07-01", features].copy()
X_train, y_train = train[features], train[target]
X_valid, y_valid = valid[features], valid[target]

assert (len(train), len(valid), len(X_test_reserved)) == (364, 182, 184)
assert train.dteday.max() < valid.dteday.min()
print("Training:", X_train.shape, "Validation:", X_valid.shape,
      "Reserved test:", X_test_reserved.shape)

In [ ]:
fig, ax = plt.subplots()
ax.plot(train.dteday, y_train, color="#2a78d6", lw=1, label="Training")
ax.plot(valid.dteday, y_valid, color="#d85b20", lw=1, label="Validation")
ax.axvspan(pd.Timestamp("2012-07-01"), pd.Timestamp("2012-12-31"),
           color="0.9", label="Reserved test; targets hidden")
ax.set(xlabel="Date", ylabel="Rentals per day", title="Demand changes over time")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

## 3. Fit, predict, measure

`DummyRegressor(strategy="mean")` learns one number: the mean training target. It predicts that number for every new input. We use explicit metrics: MAE is in rentals, and MSE is in rentals squared.

In [ ]:
model = DummyRegressor(strategy="mean")
model.fit(X_train, y_train)
y_pred = model.predict(X_valid)

print(f"Training mean: {y_train.mean():.1f} rentals")
print(f"Validation MAE: {mean_absolute_error(y_valid, y_pred):.1f} rentals")
print(f"Validation MSE: {mean_squared_error(y_valid, y_pred):.1f} rentals squared")

**Your reasoning 2.** Identify the first line where validation targets affect the calculation. Explain why `predict` receives `X_valid` but not `y_valid`.

*Write your answer here.*

## 4. Compare on the same validation observations

Fit a linear predictor using the same two inputs. Its fitting algorithm is supplied; focus on its inputs, outputs, and measured behavior. Also include a persistence forecast: predict that today’s count equals yesterday’s.

The model classes use the same `.fit()` and `.predict()` pattern. The persistence forecast is a direct rule with no fitted parameters.

In [ ]:
models = {"Training mean": DummyRegressor(strategy="mean"),
          "Fitted linear model": LinearRegression()}
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_valid)
predictions["Yesterday’s count"] = X_valid["yesterday_rentals"].to_numpy()

results = pd.DataFrame([
    {"Predictor": name,
     "MAE (rentals)": mean_absolute_error(y_valid, pred),
     "MSE (rentals²)": mean_squared_error(y_valid, pred)}
    for name, pred in predictions.items()
]).set_index("Predictor")
results.round(1)

In [ ]:
fig, ax = plt.subplots()
window = slice(0, 60)
ax.plot(valid.dteday.iloc[window], y_valid.iloc[window],
        color="#2a78d6", lw=2, label="Observed")
for name, color in [("Training mean", "#777777"),
                    ("Yesterday’s count", "#6a3fb5"),
                    ("Fitted linear model", "#d85b20")]:
    ax.plot(valid.dteday.iloc[window], predictions[name][window],
            color=color, lw=1.2, label=name)
ax.set(xlabel="Date", ylabel="Rentals per day",
       title="First 60 validation days; the error table uses all 182")
ax.legend(ncol=2)
fig.tight_layout()
plt.show()

**Your reasoning 3.** Which predictor has the lowest MAE? Which has the lowest MSE? Explain why these need not agree. State what this comparison establishes for this task and period.

*Write your answer here.*

## Decision record

Write three or four sentences identifying:

1. The target and prediction time.
2. The training/validation periods and a meaningful baseline.
3. A measured result, with metric and units.
4. A limitation and one next experiment that keeps the final test set reserved.

*Write your record here.*

## 5. Small calculations

Work out these numbers before running the cell. Four observed values are 10, 20, 30, 40. The predictions are 8, 22, 26, 44. Compute the mean residual, MAE, and MSE. Explain why mean residual alone is insufficient.

In [ ]:
observed = np.array([10., 20., 30., 40.])
predicted = np.array([8., 22., 26., 44.])
residual = observed - predicted
print("Residuals:", residual)
print("Mean residual:", residual.mean())
print("MAE:", np.abs(residual).mean())
print("MSE:", (residual ** 2).mean())

## 6. Flexibility and new observations

This is a **separate synthetic experiment**, not the bike dataset. We know the smooth signal and add independent noise to training and validation observations. The same observations are used for each candidate degree. Run the code, then compare the curves and both errors.

In [ ]:
rng = np.random.default_rng(21)
signal = lambda x: 2 + 1.3*x - 1.4*x*x + 0.8*x*x*x
x = np.linspace(-1, 1, 14)
y = signal(x) + rng.normal(0, 0.35, len(x))
xv = np.linspace(-0.95, 0.95, 50)
yv = signal(xv) + rng.normal(0, 0.35, len(xv))
grid = np.linspace(-1, 1, 201)
records = []
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, degree in zip(axes, [1, 3, 12]):
    candidate = make_pipeline(PolynomialFeatures(degree, include_bias=False),
                              LinearRegression())
    candidate.fit(x[:, None], y)
    train_error = mean_squared_error(y, candidate.predict(x[:, None]))
    valid_error = mean_squared_error(yv, candidate.predict(xv[:, None]))
    records.append({"degree": degree, "training MSE": train_error,
                    "validation MSE": valid_error})
    ax.scatter(xv, yv, s=14, marker="s", color="#d85b20", label="Validation")
    ax.scatter(x, y, s=35, color="#2a78d6", label="Training")
    ax.plot(grid, candidate.predict(grid[:, None]), color="#222222", lw=1.5)
    ax.set(title=f"Degree {degree}", xlabel="Input x", ylabel="Target y")
    ax.legend(fontsize=9)
fig.suptitle("Synthetic observations; vertical ranges vary to show each full curve")
fig.tight_layout()
plt.show()
pd.DataFrame(records).round(5)

**Your reasoning 4.** Explain why almost-zero training error can coexist with substantial validation error. Would selecting a degree using test error preserve an untouched test set?

*Write your answer here.*

## 7. A leakage check

`cnt = casual + registered` is exact in this historical file. Evaluate the identity only on training and validation rows below. Explain why it is still not a valid forecast at the start of the day.

In [ ]:
development = pd.concat([train, valid])
reconstructed = development["casual"] + development["registered"]
assert np.array_equal(reconstructed.to_numpy(), development["cnt"].to_numpy())
print("Reconstruction MAE:", mean_absolute_error(development["cnt"], reconstructed))
print("The components become available only after the day’s rentals occur.")

## References

- [Dataset and license](https://doi.org/10.24432/C5W894)
- [DummyRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html)
- [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [Validation and splitting](https://scikit-learn.org/stable/modules/cross_validation.html)
- [Data leakage and preprocessing](https://scikit-learn.org/stable/common_pitfalls.html)

The predictions in this notebook are for a teaching exercise. Model comparisons use the specified validation period; the final test period has not been scored.